In [1]:
import os

# 1. Force Hugging Face à télécharger dans le projet
os.environ["HF_HOME"] = "./hf_cache"

# 2. Force Hugging Face à ignorer le fichier de token local (qui bloque)
os.environ["HF_HUB_DISABLE_TOKEN"] = "1"

print("Dossier de cache configuré sur :", os.environ["HF_HOME"])
print("Vérification du token désactivée pour éviter l'erreur de permission.")

Dossier de cache configuré sur : ./hf_cache
Vérification du token désactivée pour éviter l'erreur de permission.


In [2]:
"""
Pipeline DistilBERT — Adversarial Training + Curriculum Learning
=======================================================================
MODIFICATIONS VS PIPELINE ORIGINAL :
  1. WeightedTrainer paramétrisé  → class_weights injectés par constructeur
  2. Chargement + oversampling 1:3 des adversarial examples (Section 2.5)
  3. Deux datasets distincts : train_normal (Phase 1) / train_full (Phase 2)
  4. Curriculum Learning en 2 phases :
       Phase 1 — lr=1.5e-5, 2 epochs  → uniquement données normales (faciles)
       Phase 2 — lr=5e-6,  2 epochs  → données normales + adversarial (difficiles)
=======================================================================
"""

import os, random, re, warnings
warnings.filterwarnings("ignore")

os.environ["TOKENIZERS_PARALLELISM"]      = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

if hasattr(torch.backends, "mps"):
    torch.backends.mps.is_available = lambda: False

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print(f"🖥️  Device : {DEVICE}")

# ======================================================================
# 0. WeightedTrainer PARAMÉTRISÉ                            ← MODIFIÉ
#    Avant : class_weights était une variable globale capturée en closure
#    Après : passé en argument du constructeur → réutilisable pour
#            les 2 phases avec des poids différents
# ======================================================================
class WeightedTrainer(Trainer):
    """Trainer avec CrossEntropyLoss pondérée pour gérer le déséquilibre de classes."""

    def __init__(self, *args, class_weights: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights  # None = pas de pondération (fallback standard)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = nn.CrossEntropyLoss(weight=self.class_weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

# ======================================================================
# 1. CHARGEMENT ET ÉQUILIBRAGE (UNDERSAMPLING)              ← inchangé
# ======================================================================
print("\n📂 Chargement des données...")
train_df = pd.read_csv('data/train_augmented.csv')
val_df   = pd.read_csv('data/val.csv')
test_df  = pd.read_csv('data/test.csv')

TARGET_COL = 'queue' if 'queue' in train_df.columns else 'label'

for df in [train_df, val_df, test_df]:
    df.dropna(subset=['text', TARGET_COL], inplace=True)

df_support  = train_df[train_df[TARGET_COL] == 'Support']
df_customer = train_df[train_df[TARGET_COL] == 'Customer Service']
df_billing  = train_df[train_df[TARGET_COL] == 'Billing and Payments']

if len(df_support) > 2000:
    df_support = df_support.sample(n=2000, random_state=SEED)
    train_df   = pd.concat([df_support, df_customer, df_billing]) \
                   .sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"📊 Distribution train (après undersampling) :\n{train_df[TARGET_COL].value_counts()}")

# ======================================================================
# 2. NETTOYAGE (VERSION TRANSFORMER)                        ← inchangé
# ======================================================================
PLACEHOLDER_RE = re.compile(r'\[.*?\]')
GREETINGS_RE   = re.compile(r'^(hello|hi|dear customer|hey)[,\s]+', flags=re.IGNORECASE)
SIGNATURE_RE   = re.compile(
    r'(sincerely|best regards|warm regards|thank you for your time).*',
    flags=re.IGNORECASE | re.DOTALL,
)
VERSION_RE  = re.compile(r'\d+\.\d+(\.\d+)?')
SOFTWARE_RE = re.compile(
    r'\b(microsoft dynamics 365|microsoft dynamics|laravel 8|laravel|node\.js|elasticsearch|'
    r'avid pro tools|microsoft office 2021|office 2021|hadoop|apache|cyberlink powerdirector|'
    r'davinci resolve|asana)\b',
    flags=re.IGNORECASE,
)
SPACES_RE = re.compile(r'\s+')


def clean_text_bert(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = GREETINGS_RE.sub('', text)
    text = SIGNATURE_RE.sub('', text)
    text = PLACEHOLDER_RE.sub('', text)
    text = VERSION_RE.sub('VERSION', text)
    text = SOFTWARE_RE.sub('SOFTWARE_PRODUCT', text)
    return SPACES_RE.sub(' ', text).strip()


label_mapping = {
    "Support":              0,
    "Feature Request":      1,
    "Billing and Payments": 2,
}


def apply_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """Applique le pipeline de nettoyage complet à un dataframe."""
    df = df.copy()
    df['text'] = df['text'].apply(clean_text_bert)
    df['text'].replace('', pd.NA, inplace=True)
    col = 'queue' if 'queue' in df.columns else 'label'
    df['label'] = df[col].map(label_mapping)
    df.dropna(subset=['text', 'label'], inplace=True)
    df['label'] = df['label'].astype(int)
    df = df[df['text'].str.split().str.len() > 2]
    return df.reset_index(drop=True)


print("🧹 Nettoyage en cours...")
train_df = apply_cleaning(train_df)
val_df   = apply_cleaning(val_df)
test_df  = apply_cleaning(test_df)
print(f"✨ Tailles — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ======================================================================
# 2.5 ⚔️  ADVERSARIAL DATA — Injection avec ratio 1:3       ← NOUVEAU
#
#  Ratio 1:3 signifie : pour chaque exemple adversarial, 3 exemples
#  normaux → les adversarial représentent 25% du dataset augmenté.
#
#  Comme on n'a que 40 exemples, on les OVERSAMPLE (replace=True)
#  jusqu'à atteindre la cible n_adv = len(train_df) // 3
#  On conserve train_df_normal pour la Phase 1 du curriculum.
# ======================================================================
print("\n⚔️  Chargement des adversarial examples...")
adv_df = pd.read_csv('data/adversarial_tickets.csv')   # ← Fichier CSV généré

# Même pipeline de nettoyage que le reste du corpus
adv_df['text']  = adv_df['text'].apply(clean_text_bert)
adv_df['label'] = adv_df['label'].map(label_mapping)
adv_df.dropna(subset=['text', 'label'], inplace=True)
adv_df['label'] = adv_df['label'].astype(int)
adv_df = adv_df[adv_df['text'].str.split().str.len() > 2].reset_index(drop=True)
print(f"   ↳ {len(adv_df)} exemples bruts chargés : {adv_df['label'].value_counts().to_dict()}")

# --- Calcul du nombre cible d'adversarial pour respecter le ratio 1:3 ---
n_adv_target    = len(train_df) // 3
adv_oversampled = adv_df.sample(n=n_adv_target, replace=True, random_state=SEED)

# --- Deux versions du train set ---
train_df_normal = train_df.copy()  # Phase 1 : données faciles uniquement
train_df_full   = (                # Phase 2 : normales + adversarial
    pd.concat([train_df, adv_oversampled])
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)

pct_adv = 100 * len(adv_oversampled) / len(train_df_full)
print(f"\n📊 Train NORMAL  (Phase 1) : {len(train_df_normal):>6} samples")
print(f"📊 Train FULL    (Phase 2) : {len(train_df_full):>6} samples")
print(f"   ↳ dont {len(adv_oversampled):>5} adversarial ({pct_adv:.1f}% du total — ratio 1:3 ✓)")

# ======================================================================
# 3. TOKENISATION — 2 datasets train créés                 ← MODIFIÉ
# ======================================================================
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 128

print(f"\n🔤 Chargement tokenizer : {MODEL_NAME}")
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN
    )


def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    """Convertit un DataFrame en HuggingFace Dataset tokenisé et formaté."""
    ds = (
        Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
               .map(tokenize, batched=True)
               .rename_column('label', 'labels')
    )
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds


print("🔢 Tokenisation des datasets...")
train_dataset_normal = make_hf_dataset(train_df_normal)  # ← Phase 1
train_dataset_full   = make_hf_dataset(train_df_full)    # ← Phase 2
val_dataset          = make_hf_dataset(val_df)
test_dataset         = make_hf_dataset(test_df)

# ======================================================================
# 4. MODÈLE                                                 ← inchangé
# ======================================================================
NUM_LABELS = 3
ID2LABEL   = {0: "Support", 1: "Feature Request", 2: "Billing and Payments"}
LABEL2ID   = {v: k for k, v in ID2LABEL.items()}

print(f"\n🧠 Chargement modèle : {MODEL_NAME}")
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
model.to(DEVICE)

# ======================================================================
# 5. 🎓 CURRICULUM LEARNING — PHASE 1 : données normales   ← NOUVEAU
#
#  Objectif : construire des représentations sémantiques solides
#  sur les tickets "propres", sans bruit adversarial.
#
#  Hyperparamètres Phase 1 :
#    - lr = 1.5e-5   (standard, le modèle apprend librement)
#    - warmup_ratio = 0.1 (10% des steps)
#    - 2 epochs max + EarlyStopping patience=2
# ======================================================================
print("\n" + "=" * 62)
print("  🎓 CURRICULUM — PHASE 1 : Données normales (faciles)")
print("=" * 62)

cw_normal = compute_class_weight(
    'balanced', classes=np.array([0, 1, 2]), y=train_df_normal['label'].values
)
cw_normal_tensor = torch.tensor(cw_normal, dtype=torch.float).to(DEVICE)
print(f"  Poids de classes Phase 1 : {dict(zip(ID2LABEL.values(), cw_normal.round(3)))}")

args_phase1 = TrainingArguments(
    output_dir                  = "./curriculum-phase1",
    num_train_epochs            = 2,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 1.5e-5,     # LR standard pour Phase 1
    weight_decay                = 0.05,
    warmup_ratio                = 0.1,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "macro_f1",
    greater_is_better           = True,
    report_to                   = "none",
    use_cpu                     = True,
)

trainer_phase1 = WeightedTrainer(
    model           = model,
    args            = args_phase1,
    train_dataset   = train_dataset_normal,   # ← DONNÉES FACILES seulement
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    class_weights   = cw_normal_tensor,       # ← Poids calculés sur normal
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\n🏃 Démarrage Phase 1...")
trainer_phase1.train()

# Sauvegarde explicite du meilleur checkpoint Phase 1
PHASE1_BEST = "./curriculum-phase1-best"
trainer_phase1.save_model(PHASE1_BEST)
print(f"\n✅ Phase 1 terminée — meilleur modèle sauvegardé dans '{PHASE1_BEST}'")

# ======================================================================
# 6. 🔥 CURRICULUM LEARNING — PHASE 2 : normal + adversarial ← NOUVEAU
#
#  Le modèle hérite des poids de la Phase 1 (load_best_model_at_end=True
#  + même objet `model` en mémoire).
#
#  Hyperparamètres Phase 2 :
#    - lr = 5e-6   (÷3 vs Phase 1 → fine-tuning doux)
#    - warmup_ratio = 0.05 (warmup plus court)
#    - class_weights recalculés sur la distribution augmentée
#
#  Pourquoi un LR plus bas ?
#    Un LR élevé en Phase 2 risque de "catastrophic forgetting" :
#    le modèle oublierait les patterns appris en Phase 1.
#    Un LR bas préserve les embeddings et ajuste seulement les
#    décisions aux frontières ambiguës que les adversarial révèlent.
# ======================================================================
print("\n" + "=" * 62)
print("  🔥 CURRICULUM — PHASE 2 : + Adversarial examples (durs)")
print("=" * 62)

cw_full = compute_class_weight(
    'balanced', classes=np.array([0, 1, 2]), y=train_df_full['label'].values
)
cw_full_tensor = torch.tensor(cw_full, dtype=torch.float).to(DEVICE)
print(f"  Poids de classes Phase 2 : {dict(zip(ID2LABEL.values(), cw_full.round(3)))}")

args_phase2 = TrainingArguments(
    output_dir                  = "./curriculum-phase2-final",
    num_train_epochs            = 2,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 5e-6,       # ← LR divisé par 3 vs Phase 1
    weight_decay                = 0.05,
    warmup_ratio                = 0.05,       # ← Warmup plus court
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "macro_f1",
    greater_is_better           = True,
    report_to                   = "none",
    use_cpu                     = True,
)

trainer_phase2 = WeightedTrainer(
    model           = model,                  # ← Même modèle, poids Phase 1 chargés
    args            = args_phase2,
    train_dataset   = train_dataset_full,     # ← DONNÉES COMPLÈTES avec adversarial
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    class_weights   = cw_full_tensor,         # ← Poids recalculés sur distribution augmentée
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\n🔥 Démarrage Phase 2...")
trainer_phase2.train()
print("\n✅ Phase 2 terminée — Curriculum Learning complet !")

# ======================================================================
# 7. ÉVALUATION FINALE                                      ← inchangé
#    On évalue avec trainer_phase2 qui porte le meilleur modèle final
# ======================================================================
print("\n📊 --- ÉVALUATION FINALE SUR LE TEST SET ---")
test_output = trainer_phase2.predict(test_dataset)
pred_labels = np.argmax(test_output.predictions, axis=-1)
true_labels = test_output.label_ids

macro_f1 = f1_score(true_labels, pred_labels, average='macro')
acc       = accuracy_score(true_labels, pred_labels)

print(f"\n🎯 Macro F1-score : {macro_f1:.4f}")
print(f"✅ Accuracy        : {acc:.4f}")
print("\n📝 --- RAPPORT DÉTAILLÉ ---")
print(classification_report(true_labels, pred_labels, target_names=list(ID2LABEL.values())))

🖥️  Device : cpu

📂 Chargement des données...
📊 Distribution train (après undersampling) :
label
Support                 1675
Billing and Payments    1121
Feature Request          816
Name: count, dtype: int64
🧹 Nettoyage en cours...
✨ Tailles — Train: 3609 | Val: 439 | Test: 441

⚔️  Chargement des adversarial examples...
   ↳ 40 exemples bruts chargés : {0: 20, 2: 20}

📊 Train NORMAL  (Phase 1) :   3609 samples
📊 Train FULL    (Phase 2) :   4812 samples
   ↳ dont  1203 adversarial (25.0% du total — ratio 1:3 ✓)

🔤 Chargement tokenizer : distilbert-base-uncased
🔢 Tokenisation des datasets...


Map:   0%|          | 0/3609 [00:00<?, ? examples/s]

Map:   0%|          | 0/4812 [00:00<?, ? examples/s]

Map:   0%|          | 0/439 [00:00<?, ? examples/s]

Map:   0%|          | 0/441 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.190859,0.915426,0.906606
2,No log,0.179686,0.925277,0.917995



✅ Phase 1 terminée — meilleur modèle sauvegardé dans './curriculum-phase1-best'

  🔥 CURRICULUM — PHASE 2 : + Adversarial examples (durs)
  Poids de classes Phase 2 : {'Support': np.float64(0.719), 'Feature Request': np.float64(1.966), 'Billing and Payments': np.float64(0.908)}

🔥 Démarrage Phase 2...


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.147894,0.924861,0.917995
2,0.154400,0.155882,0.929035,0.922551



✅ Phase 2 terminée — Curriculum Learning complet !

📊 --- ÉVALUATION FINALE SUR LE TEST SET ---



🎯 Macro F1-score : 0.9121
✅ Accuracy        : 0.9048

📝 --- RAPPORT DÉTAILLÉ ---
                      precision    recall  f1-score   support

             Support       0.89      0.91      0.90       209
     Feature Request       1.00      1.00      1.00       102
Billing and Payments       0.85      0.82      0.84       130

            accuracy                           0.90       441
           macro avg       0.91      0.91      0.91       441
        weighted avg       0.90      0.90      0.90       441

